In [1]:
import duckdb
import pandas as pd

# Connect to a fast in-memory DuckDB instance
con = duckdb.connect()

# Point to your Silver folder using a wildcard (*) to read ALL files at once
silver_path = "C:/Omnijourney_Kofking_github/data/silver/*.parquet"

print("🔥 PIPELINE VERIFICATION RUN 🔥\n")

# --- TEST 1: The Cleaning & SCD2 Proof ---
print("1. PROOF OF CLEANING (Standardized Data & SCD2 Setup):")
df_sample = con.execute(f"""
    SELECT 
        property_id, 
        city, 
        price, 
        area_marla, 
        price_per_marla, 
        agency, 
        effective_start_date, 
        is_active
    FROM '{silver_path}'
    WHERE price_per_marla IS NOT NULL
    LIMIT 5
""").df()
display(df_sample) # In Jupyter, 'display' makes beautiful tables

print("\n" + "="*50 + "\n")

# --- TEST 2: The Analytics Proof (What your UI will actually request) ---
print("2. UI ANALYTICS READY: Market Overview by City:")
df_market = con.execute(f"""
    SELECT 
        city,
        COUNT(*) as total_active_listings,
        CAST(AVG(price) AS BIGINT) as avg_price,
        CAST(AVG(price_per_marla) AS BIGINT) as avg_price_per_marla
    FROM '{silver_path}'
    WHERE is_active = TRUE
    GROUP BY city
    ORDER BY total_active_listings DESC
""").df()
display(df_market)

print("\n" + "="*50 + "\n")

# --- TEST 3: The Null Handling Proof ---
print("3. MISSING DATA HANDLED (Agents & Agencies):")
df_nulls = con.execute(f"""
    SELECT 
        agency, 
        COUNT(*) as count 
    FROM '{silver_path}' 
    GROUP BY agency 
    ORDER BY count DESC 
    LIMIT 5
""").df()
display(df_nulls)

🔥 PIPELINE VERIFICATION RUN 🔥

1. PROOF OF CLEANING (Standardized Data & SCD2 Setup):


,property_id,city,price,area_marla,price_per_marla,agency,effective_start_date,is_active
0,1,Islamabad,10000000,4.0,2.500000e+06,Direct Listing,2025-03-01,True
1,2,Islamabad,6900000,5.6,1.232143e+06,Direct Listing,2025-03-01,True
2,3,Islamabad,16500000,8.0,2.062500e+06,Direct Listing,2025-03-01,True
3,4,Islamabad,43500000,40.0,1.087500e+06,Direct Listing,2025-03-01,True
4,5,Islamabad,7000000,8.0,8.750000e+05,Easy Property,2025-03-01,True




2. UI ANALYTICS READY: Market Overview by City:


,city,total_active_listings,avg_price,avg_price_per_marla
0,Karachi,121217,19791249,1935290
1,Lahore,82940,25040952,2228946
2,Islamabad,74918,13317445,1079768
3,Rawalpindi,42054,8560630,1011392
4,Faisalabad,16270,7841203,1050596




3. MISSING DATA HANDLED (Agents & Agencies):


,agency,count
0,Direct Listing,88355
1,Real Investment Consultants,1584
2,Mash Allah Estate & Builders,1420
3,Makkah Associates,827
4,Chaudhry Estate,779


In [31]:
import psycopg2
from psycopg2.extras import RealDictCursor

DB_CONFIG = {
    "host": "db.gghdhotsmjuudujbngta.supabase.co",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "postgresking786",
    "connect_timeout": 10,
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)


In [ ]:
def run_query(query, params=None, fetch=False):
    with get_connection() as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(query, params or ())
            if fetch:
                return cur.fetchall()
            conn.commit()
            cur.close()
            return None
        

In [25]:
run_query("CREATE SCHEMA IF NOT EXISTS prop_intel;")
run_query("""
CREATE TABLE IF NOT EXISTS prop_intel.properties (
    id SERIAL PRIMARY KEY,
    name TEXT NOT NULL,
    price NUMERIC,
    created_at TIMESTAMP DEFAULT NOW()
);
""")


In [26]:
run_query(
    "INSERT INTO prop_intel.properties (name, price) VALUES (%s, %s);",
    ("Test House", 123456)
)

rows = run_query("SELECT * FROM prop_intel.properties;", fetch=True)
print(rows)

run_query("UPDATE prop_intel.properties SET price = %s WHERE id = %s;", (150000, 1))
run_query("DELETE FROM prop_intel.properties WHERE id = %s;", (1,))

[RealDictRow({'id': 2, 'name': 'Test House', 'price': Decimal('123456'), 'created_at': datetime.datetime(2026, 3, 24, 20, 32, 26, 865389)})]


In [ ]:
run_query("select * from prop_intel;")


In [27]:
conn.close()

In [ ]:
run_query("DROP TABLE IF EXISTS prop_intel.properties;")
run_query("DROP SCHEMA IF EXISTS prop_intel CASCADE;")

In [13]:
def transaction_example():
    conn = get_connection()
    try:
        with conn.cursor() as cur:
            cur.execute("INSERT INTO prop_intel.properties (name, price) VALUES (%s, %s);", ("A", 1))
            cur.execute("INSERT INTO prop_intel.properties (name, price) VALUES (%s, %s);", ("B", 2))
        conn.commit()
    except Exception as ex:
        conn.rollback()
        raise
    finally:
        conn.close()

In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor

DB_CONFIG = { ... }  # same as above

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

def run_query(query, params=None, fetch=False):
    with get_connection() as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(query, params or ())
            if fetch:
                return cur.fetchall()

# create schema/table
run_query("CREATE SCHEMA IF NOT EXISTS prop_intel;")
run_query("""
CREATE TABLE IF NOT EXISTS prop_intel.properties (
    id SERIAL PRIMARY KEY,
    name TEXT,
    price NUMERIC
);
""")

# insert + fetch
run_query("INSERT INTO prop_intel.properties (name, price) VALUES (%s, %s);", ("A", 10))
print(run_query("SELECT * FROM prop_intel.properties;", fetch=True))